# Lab 6: LLMs Inside Your Analysis
**BUS-G 492 · Coding with AI for Business Analysts**

> **INSTRUCTOR VERSION — contains solutions. Do not distribute.**

In [1]:
# --- Setup: run this cell first -------------------------------------------
# Finds the course data whether you are in Colab or on your own laptop.
# INSTRUCTOR: set DATA_URL once to the raw GitHub folder that holds /data (see instructor README).
import os
DATA_URL = "https://raw.githubusercontent.com/ilovemypinktutu/g492-coding/main/data/"
for candidate in ["data/", "../data/", "../../data/"]:
    if os.path.exists(candidate + "orders.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

Reading data from: ../../data/


**Scenario.** Limestone Outfitters has thousands of free-text product reviews. The VP of Customer Experience asks: *"What are customers actually complaining about, and is it getting better?"* Nobody is going to read 50,000 reviews.

Today you'll use an LLM as a **component inside your code**: a classifier you call thousands of times. Then you'll do the part most people skip, which is **measuring how often it's right**.

You have 114 reviews that a human already labeled (`human_topic`, `human_sentiment`). That is your *test set*.

In [2]:
import pandas as pd, json, os
reviews = pd.read_csv(DATA + "reviews.csv")
print(reviews.shape)
reviews.sample(8, random_state=1)[["review_id", "stars", "review_text", "human_topic", "human_sentiment"]]

(114, 6)


,review_id,stars,review_text,human_topic,human_sentiment
93,94,5,Bought these for my dad. He says they're 'fine...,Quality,Positive
44,45,3,It's a bucket that folds. Fine.,Quality,Mixed
56,57,5,"Ordered Tuesday, arrived Thursday. Packaging w...",Shipping,Positive
82,83,2,Straps don't adjust short enough for a 5 year ...,Fit/Sizing,Negative
69,70,2,$529 and the zippers feel like they came off a...,Value,Negative
105,106,3,Bright enough. Headband gets sweaty.,Quality,Mixed
101,102,2,Keeps drinks hot but the paint chipped off in ...,Quality,Mixed
58,59,5,Tent was defective but support sent a new one ...,Customer Service,Positive


## Part 1: The labeling guide is the spec (10 min)
An LLM classifier is only as good as its instructions. It's the same lesson as Lab 1. Read the guide below, then **improve it**: add at least two rules for cases you think are ambiguous (sarcasm? two topics in one review? non-English? "ok"?).

In [3]:
GUIDE = """You label customer reviews for an outdoor-gear retailer.

topic: the ONE main issue the review is about:
  - Quality: how the product performs or holds up
  - Fit/Sizing: size, length, how it fits the body
  - Shipping: delivery speed, damage in transit, wrong item sent
  - Customer Service: interactions with support, returns/warranty handling
  - Value: price relative to what you get

sentiment: Positive, Negative, or Mixed (clearly both good and bad points).

Rules:
- (add your rules here)
"""

## Part 2: Run the classifier (15 min)
Choose **one** path.

**Path A: API (if you have an Anthropic API key).** In Colab, click the 🔑 *Secrets* icon on the left, add `ANTHROPIC_API_KEY`, and toggle notebook access on. Never paste a key into a cell. Keys in notebooks end up on GitHub.
Classifying all 114 reviews costs well under $1.

**Path B: no key (chat assistant).** Run the Path-B cell to print the prompt in batches of 20, paste each batch into your chat assistant, and paste its CSV answers back into `pasted_csv`.

In [4]:
# ---------- Path A: API ----------
# %pip install -q anthropic
import anthropic
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass   # not in Colab: the SDK reads ANTHROPIC_API_KEY from your environment

client = anthropic.Anthropic()
MODEL = "claude-opus-5-5"
SCHEMA = {
    "type": "object",
    "properties": {
        "reason": {"type": "string", "description": "one short sentence explaining the labels"},
        "topic": {"type": "string", "enum": ["Quality", "Fit/Sizing", "Shipping", "Customer Service", "Value"]},
        "sentiment": {"type": "string", "enum": ["Positive", "Negative", "Mixed"]},
    },
    "required": ["reason", "topic", "sentiment"],
    "additionalProperties": False,
}

def classify(text, guide):
    """Label one review. Returns a dict with reason, topic, sentiment (plus token counts for cost)."""
    resp = client.beta.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=guide,
        messages=[{"role": "user", "content": f"Review:\n{text}"}],
        output_config={"effort": "low", "format": {"type": "json_schema", "schema": SCHEMA}},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",      # if the model declines, the API retries on a fallback model
    )
    if resp.stop_reason == "refusal":
        return {"reason": "refused", "topic": None, "sentiment": None}
    out = json.loads(next(b.text for b in resp.content if b.type == "text"))
    out["input_tokens"], out["output_tokens"] = resp.usage.input_tokens, resp.usage.output_tokens
    return out

print(classify("Great boots but they took three weeks to arrive.", GUIDE))

TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [5]:
# INSTRUCTOR: set MOCK=True to test the evaluation cells without an API key.
MOCK = os.environ.get("MOCK_LLM") == "1"
if MOCK:
    import numpy as np
    rng = np.random.default_rng(0)
    hard = pd.read_csv("../reviews_hard_flags.csv") if os.path.exists("../reviews_hard_flags.csv") else None
    preds = reviews[["review_id"]].copy()
    flip = rng.random(len(reviews)) < 0.15
    topics = ["Quality", "Fit/Sizing", "Shipping", "Customer Service", "Value"]
    preds["topic"] = [rng.choice(topics) if f else t for f, t in zip(flip, reviews.human_topic)]
    preds["sentiment"] = [rng.choice(["Positive", "Negative", "Mixed"]) if f else s
                          for f, s in zip(rng.random(len(reviews)) < 0.12, reviews.human_sentiment)]
    preds["reason"] = "mock"; preds["input_tokens"] = 260; preds["output_tokens"] = 45
else:
    results = [{"review_id": r.review_id, **classify(r.review_text, GUIDE)} for r in reviews.itertuples()]
    preds = pd.DataFrame(results)
preds.head()

,review_id,topic,sentiment,reason,input_tokens,output_tokens
0,1,Quality,Positive,mock,260,45
1,2,Fit/Sizing,Negative,mock,260,45
2,3,Quality,Mixed,mock,260,45
3,4,Customer Service,Positive,mock,260,45
4,5,Quality,Negative,mock,260,45


In [6]:
# ---------- Path B: no API key ----------
def print_batches(guide, size=20):
    for start in range(0, len(reviews), size):
        batch = reviews.iloc[start:start + size]
        body = "\n".join(f"{r.review_id}: {r.review_text}" for r in batch.itertuples())
        print(f"===== BATCH {start // size + 1} (copy everything below) =====")
        print(guide + "\nLabel each review below. Answer ONLY with CSV lines: review_id,topic,sentiment\n\n" + body + "\n")
# print_batches(GUIDE)

pasted_csv = """review_id,topic,sentiment
"""
# preds = pd.read_csv(io.StringIO(pasted_csv))   # uncomment after pasting (and `import io`)

## Part 3: Is it any good? (20 min)
**3a. Accuracy.** What share of topic labels and sentiment labels match the human labels?

In [7]:
ev = reviews.merge(preds, on="review_id", validate="one_to_one")
print("topic accuracy:    ", f"{(ev.topic == ev.human_topic).mean():.1%}")
print("sentiment accuracy:", f"{(ev.sentiment == ev.human_sentiment).mean():.1%}")

topic accuracy:     86.8%
sentiment accuracy: 91.2%


**3b. Where does it go wrong?** A *confusion matrix* shows which labels get mistaken for which. Rows are the human labels, columns are the model labels.

In [8]:
pd.crosstab(ev.human_topic, ev.topic, margins=True)

topic,Customer Service,Fit/Sizing,Quality,Shipping,Value,All
human_topic,,,,,,
Customer Service,8,0,0,0,2,10
Fit/Sizing,0,13,1,0,0,14
Quality,2,2,51,2,2,59
Shipping,1,1,0,11,0,13
Value,0,0,1,1,16,18
All,11,16,53,14,20,114


In [9]:
pd.crosstab(ev.human_sentiment, ev.sentiment, margins=True)

sentiment,Mixed,Negative,Positive,All
human_sentiment,,,,
Mixed,26,0,2,28
Negative,2,30,0,32
Positive,4,2,48,54
All,32,32,50,114


**3c. Read the disagreements.** For at least 8 disagreements, decide who's right: the model, the human, or "genuinely ambiguous". Human labels are not perfect truth.

In [10]:
disagree = ev[(ev.topic != ev.human_topic) | (ev.sentiment != ev.human_sentiment)]
pd.set_option("display.max_colwidth", 120)
disagree[["review_id", "review_text", "human_topic", "topic", "human_sentiment", "sentiment", "reason"]]

,review_id,review_text,human_topic,topic,human_sentiment,sentiment,reason
3,4,"Light, grippy, and my feet stayed dry on wet rock. Best trail shoe I've owned.",Quality,Customer Service,Positive,Positive,mock
11,12,"Good fabric, but the legs are cut really long. I'm 5'6"" and had to hem them.",Fit/Sizing,Quality,Mixed,Mixed,mock
13,14,$249 for a jacket that wetted out in the first real storm. Absolutely not worth it.,Value,Shipping,Negative,Negative,mock
20,21,UPF hoodie that doesn't feel like wearing a trash bag in the heat. Bought two more.,Quality,Value,Positive,Positive,mock
23,24,Set up in under five minutes in the dark. Stayed bone dry in an overnight thunderstorm.,Quality,Quality,Positive,Mixed,mock
27,28,Condensation city. Woke up with the inside of the fly dripping on my bag every night.,Quality,Quality,Negative,Mixed,mock
29,30,"Simple, light, and pitches a dozen ways. Great value compared to the big brands.",Value,Value,Positive,Mixed,mock
32,33,Shoulder strap buckle snapped on the second trip. Mid-hike. Great.,Quality,Fit/Sizing,Negative,Negative,mock
36,37,Doesn't bounce at all on runs. Holds two soft flasks and a jacket.,Quality,Quality,Positive,Negative,mock
53,54,Most comfortable camp chair I've sat in and it packs small.,Quality,Shipping,Positive,Positive,mock


In [11]:
verdicts = {
    # review_id: "model" / "human" / "ambiguous"
}

**3d. Why not just use the star rating?** Compare star ratings with human sentiment. How often would "1–2 stars = negative, 4–5 = positive" be wrong?

In [12]:
star_guess = pd.cut(reviews.stars, [0, 2, 3, 5], labels=["Negative", "Mixed", "Positive"]).astype(str)
print("stars-only sentiment accuracy:", f"{(star_guess == reviews.human_sentiment).mean():.1%}")
reviews[star_guess != reviews.human_sentiment][["stars", "review_text", "human_sentiment"]].head(8)

stars-only sentiment accuracy: 86.8%


,stars,review_text,human_sentiment
2,4,Waterproofing held up through a full day of rain. Runs about half a size small though.,Mixed
14,4,Great shell. Took 11 days to arrive when checkout said 3-5.,Mixed
16,2,Feathers leaking out of the seams from day one. Customer service did send a replacement quickly.,Mixed
31,4,Comfortable with 30 lbs. Wish the rain cover were included at this price.,Mixed
40,4,Keeps coffee hot for hours. Lid leaks slightly if tipped.,Mixed
50,4,"Comfy for side sleepers, but it crinkles loudly every time you move.",Mixed
52,3,Decent kit but half of it is bandaids. Overpriced for what's in it.,Negative
61,4,Arrived fast but the box had clearly been opened and re-taped.,Mixed


## Part 4: Improve once, and don't fool yourself (10 min)
Edit `GUIDE` based on what you saw in 3c and re-run **Part 2 and 3a**.

⚠️ **The trap:** if you tune the guide until it gets *these 114* right, your accuracy number stops meaning anything. It's like studying with the answer key. In a real project you keep a **held-out** set you never look at while tuning. Write down: how would you set that up for the 50,000 real reviews?

## Part 5: Cost and governance (5 min)

In [13]:
if "input_tokens" in preds:
    tin, tout = preds.input_tokens.mean(), preds.output_tokens.mean()
    price_in, price_out = 4.00, 20.00        # USD per million tokens for claude-opus-5-5 (check current pricing)
    per_review = (tin * price_in + tout * price_out) / 1e6
    print(f"avg tokens in/out: {tin:.0f}/{tout:.0f}  ->  ${per_review:.5f} per review, "
          f"${per_review * 50_000:,.0f} for 50,000 reviews")

avg tokens in/out: 260/45  ->  $0.00194 per review, $97 for 50,000 reviews


**Discuss:** (1) Before sending customer reviews to an outside AI provider, which questions must your company answer? (Think: personal data, contracts, data retention, the order number in review 98.) (2) At what accuracy would you trust this enough to put the numbers in front of the VP, and what would you say about the error rate?

### Submit
Your final `GUIDE`, your accuracy before and after one revision, your `verdicts` for 8 disagreements, and a 3-sentence answer to "should the VP trust this?"